### LLM : 도메인 특화 챗봇 개발 파이프라인

- 이 노트북은 **`도메인 특화 챗봇` 개발 파이프라인 예제 실습**을 수행하는 노트북입니다.

##### Llama를 활용한 도메인 특화 챗봇 개발 파이프라인 예제
   1.  한국 민사법 도메인 특화 LLM Fine-Tuning 및 성능 평가 (Cell 7개)

### 한국 민사법 도메인 특화 LLM Fine-Tuning 및 성능 평가

사전학습된 Llama 3.2 Korean Bllossom 모델(`Bllossom/llama-3.2-Korean-Bllossom-AICA-5B`)을 AI Hub 민사법 데이터셋으로 Fine-tuning합니다.

법률 전문 AI 어시스턴트 구축을 위해 데이터 전처리부터 모델 학습, 평가까지 전체 파이프라인을 구현하여 한국 법률 질의응답 성능을 향상시킵니다.

* AI Hub 민사법 JSON 데이터를 로드하고 법률 텍스트 특화 전처리 (조항 번호 정규화, 날짜 형식 통일 등) 수행
* 4비트 양자화로 메모리 효율적인 모델 로딩 및 LoRA를 통한 파라미터 효율적 학습 설정
* Llama 프롬프트 템플릿 적용하여 시스템-사용자-어시스턴트 대화 형식으로 데이터 구성
* SFTTrainer를 사용하여 답변 부분만 선택적으로 학습하는 Supervised Fine-tuning 실행
* ROUGE 스코어 및 법률 용어 정확도 측정으로 다각도 성능 평가 수행

In [10]:
# ============================================
# Cell 1: 라이브러리 임포트 및 환경 설정
# ============================================

# 필수 라이브러리 임포트
import torch
import transformers
import peft
import os
import warnings
warnings.filterwarnings('ignore')

# PyTorch 및 CUDA 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Transformers 버전: {transformers.__version__}")
print(f"PEFT 버전: {peft.__version__}")

# GPU 연산 속도 최적화를 위한 설정
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # 비동기 CUDA 실행 활성화
torch.backends.cuda.matmul.allow_tf32 = True  # TensorFloat-32 연산 허용
torch.backends.cudnn.allow_tf32 = True  # cuDNN에서 TF32 사용
torch.backends.cudnn.benchmark = True  # 최적의 알고리즘 자동 선택


PyTorch 버전: 2.7.1+cu128
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 4050 Laptop GPU
GPU 메모리: 6.00 GB
Transformers 버전: 4.46.3
PEFT 버전: 0.12.0


In [11]:
# ============================================
# Cell 2: 데이터 로드 및 전처리 함수 정의
# ============================================

import json
import re
from datasets import Dataset as HFDataset

def load_legal_data(data_dir):
    """
    AI Hub 민사법 데이터를 로드하고 전처리하는 함수
    
    Args:
        data_dir (str): 데이터 디렉토리 경로
    
    Returns:
        list: 전처리된 데이터 리스트
    """
    all_data = []
    
    # 질의응답 폴더 내의 모든 JSON 파일 순회
    for root, dirs, files in os.walk(data_dir):
        if "질의응답" in root:
            for file in files:
                if file.endswith('.json'):
                    with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        
                        # 문맥 정보 추출 (최대 10개 문장)
                        context = " ".join(data['taskinfo']['sentences'][:10])
                        context = preprocess_legal_text(context)
                        question = preprocess_legal_text(data['taskinfo']['input'])
                        answer = preprocess_legal_text(data['taskinfo']['output'])
                        
                        # 학습용 데이터 구성
                        all_data.append({
                            'messages': format_prompt_template(question, context, answer),
                            'question': question,
                            'answer': answer
                        })
    
    return all_data

def preprocess_legal_text(text):
    """
    법률 텍스트 전처리 함수
    - 조항 번호, 날짜, 금액 형식 정규화
    
    Args:
        text (str): 원본 텍스트
    
    Returns:
        str: 전처리된 텍스트
    """
    # 조항 번호 정규화 (제 1 조 -> 제1조)
    text = re.sub(r'제\s*(\d+)\s*조', r'제\1조', text)
    text = re.sub(r'제\s*(\d+)\s*항', r'제\1항', text)
    
    # 날짜 형식 통일 (2024.1.1 -> 2024년 1월 1일)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', r'\1년 \2월 \3일', text)
    
    # 금액 표기의 쉼표 제거 (1,000,000 -> 1000000)
    text = re.sub(r'(\d{1,3})(,\d{3})+', lambda m: m.group(0).replace(',', ''), text)
    
    # 연속된 공백을 단일 공백으로 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def extract_legal_entities(text):
    """
    법률 텍스트에서 주요 개체 추출
    
    Args:
        text (str): 분석할 텍스트
    
    Returns:
        dict: 추출된 법률 개체들 (법원, 사건번호, 법령)
    """
    return {
        'courts': re.findall(r'[\w]*법원', text),  # 법원명 추출
        'case_numbers': re.findall(r'\d{4}[가-힣]+\d+', text),  # 사건번호 추출
        'laws': re.findall(r'[\w\s]+법(?:\s*제\d+조)?', text)[:5]  # 법령명 추출 (최대 5개)
    }

def format_prompt_template(question, context=None, answer=None):
    """
    Llama 모델용 채팅 형식 프롬프트 템플릿 생성
    
    Args:
        question (str): 사용자 질문
        context (str, optional): 참고 문서
        answer (str, optional): 정답 (학습시에만 사용)
    
    Returns:
        list: 메시지 형식의 프롬프트
    """
    # 시스템 프롬프트 설정
    system_message = "당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요."
    
    # 참고 문서가 있는 경우 포함
    if context:
        user_message = f"다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n{context}\n\n[질문]\n{question}"
    else:
        user_message = question
    
    # 학습용 (답변 포함)
    if answer:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer}
        ]
    # 추론용 (답변 미포함)  
    else:
        messages = [
            {
                "role": "system", 
                "content": system_message
            },
            {
                "role": "user", 
                "content": [
                    {
                        'type': 'text', 
                        'text': user_message
                    }
                ]
            }
        ]
    
    return messages

# 데이터 로드 및 전처리 실행
print("데이터 전처리 중...")
train_data = load_legal_data(r"C:\Users\SSAFY\Downloads\01.민사법 LLM 사전학습 및 Instruction Tuning 데이터\3.개방데이터\1.데이터\Training\02.라벨링데이터")
val_data = load_legal_data(r"C:\Users\SSAFY\Downloads\01.민사법 LLM 사전학습 및 Instruction Tuning 데이터\3.개방데이터\1.데이터\Validation\02.라벨링데이터")

# 메모리 및 학습 시간 고려하여 샘플 수 제한
max_train_samples = 1000  # 필요에 따라 조정
max_eval_samples = 100

print(f"전체 학습 데이터: {len(train_data)}")
print(f"전체 검증 데이터: {len(val_data)}")

# 샘플링
train_data = train_data[:max_train_samples]
val_data = val_data[:max_eval_samples]

print(f"전처리 완료!")
print(f"샘플 학습 데이터: {len(train_data)}")
print(f"샘플 검증 데이터: {len(val_data)}")

# 전처리 결과 확인
print("\n전처리된 샘플:")
sample = train_data[0]
print(f"질문: {sample['question']}")
print(f"답변: {sample['answer']}")
print(f"학습 데이터: {sample['messages'][:100]}...")

# HuggingFace Dataset 형식으로 변환
train_dataset = HFDataset.from_list(train_data)
val_dataset = HFDataset.from_list(val_data)

데이터 전처리 중...
전체 학습 데이터: 73065
전체 검증 데이터: 9135
전처리 완료!
샘플 학습 데이터: 1000
샘플 검증 데이터: 100

전처리된 샘플:
질문: 계약상 채무자가 계약을 이행하지 않겠다는 의사를 분명히 표시한 경우, 채권자는 어떤 법적 조치를 취할 수 있나요?
답변: 계약상 채무자가 계약을 이행하지 않겠다는 의사를 명백히 표시하면, 채권자는 신의성실의 원칙에 따라 이행기 전이라도 이행의 최고 없이 계약을 해제하거나 손해배상을 청구할 수 있습니다. 이행거절의 여부는 계약 이행에 관한 당사자의 행동과 계약 전후의 구체적인 사정을 종합적으로 판단해야 합니다.
학습 데이터: [{'role': 'system', 'content': '당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요.'}, {'role': 'user', 'content': "다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n1. 제1심 판결 중 아래에서 추가로 지급을 명하는 금원에 해당하는 원고 패소부분을 취소한다. 피고는 원고에게 86880074원 및 이에 대하여 2012. 12. 18.부터 2015. 4. 17.까지는 연 5%의, 그 다음날부터 다 갚는 날까지는 연 20%의 각 비율로 계산한 금원을 지급하라. 2. 원고의 나머지 항소와 피고의 항소를 각 기각한다. 3. 소송총비용 중 1/4은 원고가, 나머지는 피고가 각 부담한다. 4. 제1항 중 금원지급 부분은 가집행할 수 있다. 5. 제1심 판결의 주문 제1항 중 '2012. 2. 18'을 '2012. 12. 18.'로 경정한다. 피고는 원고에게 361787580원 및 이에 대하여 소장 송달일 다음날부터 다 갚는 날까 지연 20%의 비율에 의한 금원을 지급하라. 가. 원고 : 제1심 판결 중 아래에서 지급을 명하는 부분에 해당하는 원고 패소부분을 취소한다. 피고는 원고에게 176206872원 및 이에 대하여 소장 송달일 다음날부터 다 갚는

In [12]:
# ============================================
# Cell 3: 모델 로드 및 초기 설정
# ============================================
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
from transformers import (
    MllamaForConditionalGeneration,
    MllamaProcessor,
    BitsAndBytesConfig
)

# 사용할 한국어 LLM 모델
model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"
print(f"선택된 모델: {model_name}")

# 4비트 양자화 설정으로 메모리 효율성 향상
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4비트로 모델 로드
    bnb_4bit_quant_type="nf4",  # Normal Float 4 양자화
    bnb_4bit_compute_dtype=torch.float16,  # 연산은 float16으로
    bnb_4bit_use_double_quant=True,  # 이중 양자화로 추가 압축
)

# 모델 로드 (양자화 적용)
print("모델 로딩 중... (몇 분 소요될 수 있습니다)")
model = MllamaForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # GPU 자동 할당
    torch_dtype=torch.float16,  # 메모리 절약을 위한 half precision
    attn_implementation="eager",
    low_cpu_mem_usage=True,  # CPU 메모리 절약
    offload_folder="./offload"  # 디스크 오프로딩  
)

# 프로세서 및 토크나이저 설정
processor = MllamaProcessor.from_pretrained(model_name)
tokenizer = processor.tokenizer

# 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 오른쪽 패딩

print("모델 및 토크나이저 로드 완료!")
print(f"모델 크기: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B 파라미터")

# 모델 초기 테스트
test_context = "서울특별시는 대한민국의 수도이며, 인구는 약 950만 명입니다."
test_question = "한국의 수도는 어디인가요?"
template = format_prompt_template(context=test_context, question=test_question)

# 채팅 템플릿 적용
test_prompt = processor.apply_chat_template(
    template, 
    tokenize=False, 
    add_generation_prompt=True
)

# 토크나이징 및 디바이스 이동
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print("초기 추론 테스트:")
print(f"질문: {test_question}")

# 추론 실행
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# 생성된 답변 디코딩
response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"답변: {response}")

선택된 모델: Bllossom/llama-3.2-Korean-Bllossom-AICA-5B
모델 로딩 중... (몇 분 소요될 수 있습니다)


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\SSAFY\.conda\envs\llm\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\SSAFY\.conda\envs\llm\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\SSAFY\.conda\envs\llm\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\SSAFY\.conda\envs\llm\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\SSAFY\.conda\envs\llm\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 3/3 [00:08<00:00,  2.88s/it]


모델 및 토크나이저 로드 완료!
모델 크기: 3.03B 파라미터
초기 추론 테스트:
질문: 한국의 수도는 어디인가요?
답변: 한국의 수도는 서울입니다. 서울은 대한민국의 정치, 경제, 문화의 중심지로, 서울특별시는 약 950만 명의 인구를 가지고 있습니다.


In [15]:
# ============================================
# Cell 4: LoRA 설정 및 학습 준비
# ============================================

# LoRA (Low-Rank Adaptation) 설정
lora_config = LoraConfig(
    r=8,  # LoRA rank
    lora_alpha=16,  # LoRA scaling parameter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",  # Attention 모듈
                    "gate_proj", "up_proj", "down_proj"],  # MLP 모듈
    lora_dropout=0.05,  # 드롭아웃 비율
    bias="none",  # 바이어스 학습 안함
    task_type="CAUSAL_LM",  # Causal Language Modeling
)

# SFT (Supervised Fine-Tuning) 학습 설정
training_args = SFTConfig(
    output_dir="./legal-chatbot",  # 모델 저장 경로
    num_train_epochs=1,  # 학습 에폭 수
    per_device_train_batch_size=1,  # 배치 크기
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,     #메모리 절감
    gradient_accumulation_steps=8,  # 그래디언트 누적
    optim="paged_adamw_8bit",  # 최적화 알고리즘
    logging_steps=10,  # 로깅 주기
    logging_first_step=True,  
    logging_strategy="steps",
    learning_rate=2e-4,  # 학습률
    warmup_steps=100,  # 웜업 스텝
    save_strategy="steps",  # 저장 전략
    save_steps=200,         # 저장 스텝
    eval_strategy="steps",  # 평가 전략
    eval_steps=50,          # 평가 스텝
    fp16=True,  # 16비트 부동소수점 사용
    max_seq_length=512,  # 최대 시퀀스 길이
    packing=False,  # 시퀀스 패킹 비활성화
    report_to="none",  # 로깅 플랫폼 (wandb 등 사용 가능)
    max_grad_norm=1.0,  # 그래디언트 클리핑
    seed=42,  # 재현성을 위한 시드
    dataloader_num_workers=4,  # 데이터로더 워커 수
    dataloader_pin_memory=True,  # GPU 메모리 고정
    dataset_num_proc=4,  # 데이터셋 전처리 병렬화
    neftune_noise_alpha=5,  # NEFTune 노이즈로 학습 안정성 향상
)

print("학습 설정 완료!")

# 데이터셋 확인
print("학습 데이터셋 최종 샘플 출력")
print(train_dataset[0])
print(val_dataset[0])

# SFTTrainer 초기화
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,  # 토크나이저 전달
    peft_config=lora_config,  # LoRA 설정
)

print("Fine-tuning 시작!")

# 학습 시작
trainer.train()

print("Fine-tuning 완료!")

# 학습된 LoRA 어댑터 저장
model.save_pretrained("./legal-chatbot-lora-adapter")
processor.save_pretrained("./legal-chatbot-lora-adapter")

print("모델 저장 완료!")
print("저장 위치: ./legal-chatbot-lora-adapter")

학습 설정 완료!
학습 데이터셋 최종 샘플 출력
{'messages': [{'content': '당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요.', 'role': 'system'}, {'content': "다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n1. 제1심 판결 중 아래에서 추가로 지급을 명하는 금원에 해당하는 원고 패소부분을 취소한다. 피고는 원고에게 86880074원 및 이에 대하여 2012. 12. 18.부터 2015. 4. 17.까지는 연 5%의, 그 다음날부터 다 갚는 날까지는 연 20%의 각 비율로 계산한 금원을 지급하라. 2. 원고의 나머지 항소와 피고의 항소를 각 기각한다. 3. 소송총비용 중 1/4은 원고가, 나머지는 피고가 각 부담한다. 4. 제1항 중 금원지급 부분은 가집행할 수 있다. 5. 제1심 판결의 주문 제1항 중 '2012. 2. 18'을 '2012. 12. 18.'로 경정한다. 피고는 원고에게 361787580원 및 이에 대하여 소장 송달일 다음날부터 다 갚는 날까 지연 20%의 비율에 의한 금원을 지급하라. 가. 원고 : 제1심 판결 중 아래에서 지급을 명하는 부분에 해당하는 원고 패소부분을 취소한다. 피고는 원고에게 176206872원 및 이에 대하여 소장 송달일 다음날부터 다 갚는 날까지 연 20%의 비율에 의한 금원을 지급하라. 나. 피고 : 제1심 판결 중 피고 패소부분을 취소하고, 위 취소부분에 해당하는 원고의 청구를 기각한다. 1. 기초사실\n\n[질문]\n계약상 채무자가 계약을 이행하지 않겠다는 의사를 분명히 표시한 경우, 채권자는 어떤 법적 조치를 취할 수 있나요?", 'role': 'user'}, {'content': '계약상 채무자가 계약을 이행하지 않겠다는 의사를 명백히 표시하면, 채권자는 신의성실의 원칙에 따라 이행기 전이라도 이행의 최고 없이 계약을 해제하거나 손해배상을 청구할 수 있습니다. 이행거절의 여부는 계

Map (num_proc=4): 100%|██████████| 100/100 [00:08<00:00, 11.40 examples/s]


Fine-tuning 시작!


  1%|          | 1/125 [00:31<1:04:52, 31.39s/it]

{'loss': 3.4506, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 0.01}


...

{'loss': 1.1507, 'grad_norm': 8.134255409240723, 'learning_rate': 0.00012800000000000002, 'epoch': 0.88}


 96%|█████████▌| 120/125 [33:16<01:16, 15.30s/it]

{'loss': 1.1777, 'grad_norm': 7.364262580871582, 'learning_rate': 4.8e-05, 'epoch': 0.96}


100%|██████████| 125/125 [34:37<00:00, 16.62s/it]


{'train_runtime': 2077.2651, 'train_samples_per_second': 0.481, 'train_steps_per_second': 0.06, 'train_loss': 1.8316298751831055, 'epoch': 1.0}
Fine-tuning 완료!
모델 저장 완료!
저장 위치: ./legal-chatbot-lora-adapter


In [16]:
# ============================================
# Cell 5: 추론 함수 정의
# ============================================

def generate_legal_response(model, tokenizer, question, context=None, max_length: int = 512):
    """
    법률 질문에 대한 답변 생성 및 후처리
    
    Args:
        model: 학습된 모델
        tokenizer: 토크나이저
        question (str): 사용자 질문
        context (str, optional): 참고 문서
        max_length (int): 최대 생성 토큰 수
    
    Returns:
        tuple: (답변 텍스트, 추출된 법률 개체)
    """
    
    # 입력 텍스트 전처리
    question = preprocess_legal_text(question)
    if context:
        context = preprocess_legal_text(context)
    
    # 프롬프트 구성
    messages = format_prompt_template(question, context)
    
    # 채팅 템플릿 적용
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 텍스트 전용 처리 (멀티모달 모델이지만 텍스트만 사용)
    inputs = processor(
        text=prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 생성 설정
    generation_config = {
        'max_new_tokens': max_length,
        'temperature': 0.7,  # 창의성 제어
        'top_p': 0.9,  # nucleus sampling
        'top_k': 50,  # top-k sampling
        'repetition_penalty': 1.1,  # 반복 억제
        'no_repeat_ngram_size': 3,  # n-gram 반복 방지
        'do_sample': True,  # 샘플링 활성화
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
    }
    
    # 답변 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_config
        )
    
    # 생성된 부분만 추출하여 디코딩
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # 후처리: 중복 문장 제거
    response = response.strip()
    sentences = response.split('.')
    unique_sentences = []
    for sent in sentences:
        if sent.strip() and sent.strip() not in unique_sentences:
            unique_sentences.append(sent.strip())
    
    # 문장 재구성
    response = '. '.join(unique_sentences)
    if response and not response.endswith('.'):
        response += '.'
    
    # 법률 개체 추출
    entities = extract_legal_entities(response)
    
    # 관련 법령 및 판례 정보 추가
    if entities['laws']:
        response += f"\n\n[관련 법령: {', '.join(entities['laws'][:3])}]"
    if entities['case_numbers']:
        response += f"\n[관련 판례: {', '.join(entities['case_numbers'][:3])}]"
    
    return response, entities

In [17]:
# ============================================
# Cell 6: 모델 평가 함수 정의 및 실행
# ============================================

from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np

def evaluate_legal_chatbot(model, tokenizer, test_data, num_samples=50):
    """
    법률 챗봇 성능 평가
    - ROUGE 스코어 계산
    - 법률 용어 정확도 측정
    
    Args:
        model: 평가할 모델
        tokenizer: 토크나이저
        test_data: 평가 데이터
        num_samples (int): 평가할 샘플 수
    
    Returns:
        dict: 평가 메트릭 결과
    """
    
    # ROUGE 스코어 계산기 초기화
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    results = {'rouge1': [], 'rouge2': [], 'rougeL': [], 'legal_precision': []}
    
    # 무작위 샘플링
    indices = np.random.choice(len(test_data), min(num_samples, len(test_data)))
    
    # 각 샘플에 대해 평가
    for idx in tqdm(indices, desc="평가 중"):
        item = test_data[idx]
        question = item['question']
        reference = item['answer']  # 정답
        
        # 모델 예측 생성
        prediction, pred_entities = generate_legal_response(model, tokenizer, question)
        
        # ROUGE 점수 계산
        scores = rouge.score(reference, prediction)
        for metric in ['rouge1', 'rouge2', 'rougeL']:
            results[metric].append(scores[metric].fmeasure)
        
        # 법률 용어 정확도 계산
        ref_entities = extract_legal_entities(reference)
        pred_terms = set(sum(pred_entities.values(), []))
        ref_terms = set(sum(ref_entities.values(), []))
        
        # 정밀도 계산
        if ref_terms:
            precision = len(pred_terms & ref_terms) / len(ref_terms)
            results['legal_precision'].append(precision)
    
    return results

print("모델 평가 중...")
# 평가 실행
results = evaluate_legal_chatbot(model, tokenizer, val_data)

# 평가 결과 출력
print("\n=== 평가 결과 ===")
for metric, scores in results.items():
    if scores:
        print(f"{metric}: {np.mean(scores):.4f} (±{np.std(scores):.4f})")

모델 평가 중...


평가 중: 100%|██████████| 50/50 [1:19:13<00:00, 95.06s/it] 


=== 평가 결과 ===
rouge1: 0.0599 (±0.1280)
rouge2: 0.0000 (±0.0000)
rougeL: 0.0599 (±0.1280)
legal_precision: 0.0000 (±0.0000)


In [18]:
# ============================================
# Cell 7: 샘플 테스트 및 메모리 정리
# ============================================

# 테스트할 질문 샘플
questions = [
    "계약 해제 시 손해배상을 청구할 수 있나요?",
    "부당이득반환청구권의 성립 요건은 무엇인가요?",
    "공사대금 채권의 소멸시효는 얼마인가요?"
]

# 모델을 평가 모드로 전환
model.eval()

# 각 질문에 대한 답변 생성
print("\n예측 샘플:")
for question in questions:
    response, entities = generate_legal_response(model, tokenizer, question)
    print(f"\n질문: {question}")
    print(f"답변: {response}")
    if any(entities.values()):
        print(f"추출된 법률 개체: {entities}")

# GPU 메모리 정리
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
    
torch.cuda.empty_cache()

print("GPU 메모리 정리 완료!")
print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


예측 샘플:

질문: 계약 해제 시 손해배상을 청구할 수 있나요?
답변: 네, 계약 해제로 인한 손해 배상 청구는 가능합니다. 그러나 계약 해소를 원인으로 하는 사정이 성립해야만 합니다. 예를 들어, 계약의 조항에 따라 이행이 지연되거나 계약의 원칙적으로 정상적으로 이행되지 못하는 경우(예: 계약 당사자들이 합의 없이 계약을 이행하지 않음, 계약서에 명시된 조건이 이루어지지 않았던 경우 등)에는 계약 해소가 성립하였고, 그로 인하여 손해를 입은 사람들은 계약 해체 후에 손해금을 청구하는 것이 일반적입니다. 하지만 계약 당시에 손해 발생 시 이를 피할 수 있는 방법이나 대책이 명시되어 있다면, 그 내용에 따라 손해 금액은 부정하게 인정되지 않을 수 있습니다. 예: 계약서에서 계약 당사가 계약을 해제하면 손해의 책임이 전부가 되는 것이라고 명시한 경우, 그러한 경우에는 손해액을 부정하거나 원치 않는다고 판단하여 부과하지 않게 됩니다. 따라서 계약을 제대로 이행하려는 의사가 위 조건을 준수하지 않으면 계약 당사의 책임에 따라 해당 손해에 대한 보상이 필요하다면 계약 해지를 통해 원인과 결과를 명확히 밝혀야 하며, 그 손해 여부는 계약 당사를 포함한 다양한 사실들을 종합적으로 평가하여 결정됩니다.

[관련 법령:  하지만 계약 당시에 손해 발생 시 이를 피할 수 있는 방법]
추출된 법률 개체: {'courts': [], 'case_numbers': [], 'laws': [' 하지만 계약 당시에 손해 발생 시 이를 피할 수 있는 방법']}

질문: 부당이득반환청구권의 성립 요건은 무엇인가요?
답변: 다음 조건을 모두 만족하면 부당이익을 지급받기 위한 청구권이 생깁니다. 1. **상대방에게 미지급된 부당한 이익**: 상대방이 원고에게 부당하게 이익(금액)을 지급하거나 부당할 것을 약속하여 지급하지 않은 경우, 이를 환불할 수 있습니다. 2. **자기자연법과 계약법에서 정하는 보상의 범위 내에서 지급될 수 있는 이익** : 부당불이득이득을 회수하기 위해서는 그 이득의 범